# 15 · Features suite (every role, end to end)

Walks through GEETA as real people would, on a throwaway copy (port 8099, temporary database and
outbox): admin setup → parents/tutors/children → kid picture + PIN sign-in → world buddy → tutoring
(sums, measurement, money, shapes) → session PDF → progress → pilot tools → tonight's parent and
technical emails (in the test outbox) → email on/off → password change → deleting a child.
Uses the real model. Results: `outputs/tests/features.json`.

In [ ]:
# ---- throwaway GEETA server: own temp database + outbox, port 8099 (never the real data) ----
import os, re, sys, json, time, signal, struct, zlib, socket, tempfile, subprocess, pathlib
import urllib.request, urllib.error, http.cookiejar

ROOT = pathlib.Path.cwd()
while not (ROOT / "PROJECT_BRIEF.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
PORT = 8099
B = f"http://127.0.0.1:{PORT}"
TMP = pathlib.Path(tempfile.mkdtemp(prefix="geeta_test_"))
subprocess.run(f"lsof -ti tcp:{PORT} | xargs kill 2>/dev/null", shell=True)
time.sleep(1)
_env = dict(os.environ, GEETA_PORTAL_DIR=str(TMP / "portal"), GEETA_PORT=str(PORT), GEETA_OUTBOX_DIR=str(TMP / "outbox"))
SERVER = subprocess.Popen(["./webapp/run_webapp.sh"], env=_env, stdout=open(TMP / "server.log", "w"),
                          stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen(B + "/health", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("test server didn't start - see " + str(TMP / "server.log"))

def stop_server():
    try:
        os.killpg(SERVER.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

class _NoRedir(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *a, **k):
        return None

def client():
    return urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()), _NoRedir)

def call(op, method, path, body=None, headers=None, raw=None, ctype="application/json", timeout=120):
    data = raw if raw is not None else (json.dumps(body).encode() if body is not None else None)
    h = {"Content-Type": ctype} if data is not None else {}
    h.update(headers or {})
    req = urllib.request.Request(B + path, data=data, method=method, headers=h)
    try:
        r = op.open(req, timeout=timeout); return r.status, r.read(), dict(r.headers)
    except urllib.error.HTTPError as e:
        loc = e.headers.get("location")
        return e.code, (("-> " + loc).encode() if loc else e.read()), dict(e.headers)
    except Exception as e:
        return 0, str(e).encode(), {}

def js(res):
    try:
        return json.loads(res[1])
    except Exception:
        return {}

RESULTS = []
def check(section, name, ok, detail=""):
    RESULTS.append({"section": section, "check": name, "passed": bool(ok), "detail": str(detail)[:160]})
    print(("PASS " if ok else "FAIL ") + f"[{section}] {name:58s} {str(detail)[:90]}")

def server_tracebacks():
    return (TMP / "server.log").read_text(errors="ignore").count("Traceback")


In [ ]:
try:
    S = "pages"
    anon = client()
    for p in ("/login.html", "/parent-login.html", "/staff-login.html", "/kid.html", "/privacy.html"):
        check(S, f"{p} opens", call(anon, "GET", p)[0] == 200)
    for p in ("/static/app.js", "/static/portal.js", "/static/worlds.js", "/static/kid-world.js", "/static/worlds.css", "/static/kid.js"):
        check(S, f"{p} served", call(anon, "GET", p)[0] == 200)

    S = "admin"
    admin = client()
    r = call(admin, "POST", "/api/auth/setup", {"name": "Kiruthik", "email": "admin@test.local", "password": "AdminPass123"})
    check(S, "admin setup -> admin home", b"/admin.html" in r[1])
    pid = js(call(admin, "POST", "/api/users", {"name": "Mum", "email": "mum@test.local", "role": "parent", "password": "ParentPass9"}))["user"]["id"]
    tid = js(call(admin, "POST", "/api/users", {"name": "Tess", "email": "tess@test.local", "role": "tutor", "password": "TutorPass99"}))["user"]["id"]
    kid = js(call(admin, "POST", "/api/kids", {"nickname": "Mia", "year_level": 2, "parent_id": pid, "tutor_id": tid}))["kid"]["id"]
    check(S, "admin adds parent, tutor, child", kid > 0)
    ov = js(call(admin, "GET", "/api/admin/overview")).get("counts", {})
    check(S, "admin overview counts", ov.get("kids") == 1 and ov.get("parents") == 1 and ov.get("tutors") == 2, ov)
    check(S, "admin opens admin page", call(admin, "GET", "/admin.html")[0] == 200)
    check(S, "admin resets a password", call(admin, "POST", f"/api/users/{pid}/password", {"password": "ParentPass9"})[0] == 200)

    S = "parent"
    par = client()
    check(S, "parent can't use the staff sign-in", call(par, "POST", "/api/auth/login", {"email": "mum@test.local", "password": "ParentPass9", "expect": "staff"})[0] == 403)
    check(S, "parent signs in -> parent home", b"/parent.html" in call(par, "POST", "/api/auth/login", {"email": "mum@test.local", "password": "ParentPass9", "expect": "parent"})[1])
    check(S, "parent sees their child", any(k["nickname"] == "Mia" for k in js(call(par, "GET", "/api/kids"))["kids"]))
    check(S, "parent picks avatar + world", call(par, "PATCH", f"/api/kids/{kid}", {"avatar": "🦊", "world": "car"})[0] == 200)
    check(S, "parent sets PIN", call(par, "POST", f"/api/kids/{kid}/pin", {"pin": "4826"})[0] == 200)
    check(S, "daily email is on by default", js(call(par, "GET", "/api/auth/me"))["user"]["email_reports"] is True)

    S = "tutor"
    tut = client()
    check(S, "tutor signs in -> tutor home", b"/tutor.html" in call(tut, "POST", "/api/auth/login", {"email": "tess@test.local", "password": "TutorPass99", "expect": "staff"})[1])
    check(S, "tutor sees their student", any(k["nickname"] == "Mia" for k in js(call(tut, "GET", "/api/kids"))["kids"]))

    S = "kid"
    dev = client()
    call(dev, "POST", "/api/auth/login", {"email": "mum@test.local", "password": "ParentPass9", "expect": "parent"})
    check(S, "parent switches on kid sign-in for the device", call(dev, "POST", "/api/kid/device")[0] == 200)
    roster = js(call(dev, "GET", "/api/kid/roster")).get("kids", [])
    check(S, "kid roster shows picture, never the PIN hash", roster and roster[0]["avatar"] == "🦊" and "pin" not in json.dumps(roster).replace("has_pin", ""))
    check(S, "wrong PIN refused", call(dev, "POST", "/api/kid/login", {"kid_id": kid, "pin": "0001"})[0] == 401)
    check(S, "right PIN signs the kid in", call(dev, "POST", "/api/kid/login", {"kid_id": kid, "pin": "4826"})[0] == 200)
    sess = js(call(dev, "POST", "/api/session", {}))
    check(S, "kid session uses their world buddy (Zoom)", sess.get("kid_mode") and sess.get("kid", {}).get("world") == "car", sess.get("kid"))
    sid = sess["sid"]
    replies = [js(call(dev, "POST", f"/api/session/{sid}/text", {"text": t})).get("reply", "") for t in
               ("hi", "What is 27 + 15?", "32", "help", "just tell me the answer", "42")]
    check(S, "kid gets a friendly hello", "hi" in replies[0].lower() or "hey" in replies[0].lower(), replies[0][:60])
    check(S, "tutoring: exact first step", "27 + 10" in replies[1], replies[1][:70])
    check(S, "tutoring: wrong answer explained kindly", "close" in replies[2].lower() or "tens" in replies[2].lower(), replies[2][:70])
    check(S, "tutoring: no spoilers", "42" not in replies[4], replies[4][:70])
    check(S, "tutoring: right answer celebrated", "42 is the answer" in replies[5], replies[5][:70])
    check(S, "kid's 'I'm done' saves without showing the report", js(call(dev, "POST", f"/api/session/{sid}/close")) == {"report": None, "saved": True})
    check(S, "kid can change world", call(dev, "POST", "/api/kid/world", {"world": "dino"})[0] == 200)
    check(S, "kid signs out", call(dev, "POST", "/api/kid/logout")[0] == 200)

    S = "worksheet-by-text"
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    rr = [js(call(par, "POST", f"/api/session/{sid}/text", {"text": t})).get("reply", "") for t in
          ("A rope is 30 cm long. I cut off 12 cm. How long is it now?", "42", "18 cm",
           "How much change from $2 if I spend $1.50?", "50c", "Which shape has 3 corners?", "square", "triangle")]
    check(S, "measurement: added-instead-of-took-away spotted", "taking away" in rr[1].lower(), rr[1][:70])
    check(S, "measurement: answer with units", "18 cm is the answer" in rr[2], rr[2][:70])
    check(S, "money change: counts up from the cost", "count up" in rr[3].lower(), rr[3][:70])
    check(S, "money change: 50c accepted", "50c is the answer" in rr[4], rr[4][:70])
    check(S, "shapes: square explained, triangle accepted", "4 corners" in rr[6] and "triangle is the answer" in rr[7])
    close = js(call(par, "POST", f"/api/session/{sid}/close"))
    pdf = call(par, "GET", close["report"]["pdf_url"])[1] if close.get("report") else b""
    check(S, "parent gets a session PDF", pdf.startswith(b"%PDF"), f"{len(pdf)} bytes")

    S = "progress"
    pr = js(call(par, "GET", f"/api/kids/{kid}/progress"))
    check(S, "progress lists sessions with PDFs", len(pr.get("sessions", [])) >= 2 and any(s.get("pdf_url") for s in pr["sessions"]))
    check(S, "progress lists skills", len(pr.get("progress", {}).get("skills", [])) >= 3, [s["skill"] for s in pr.get("progress", {}).get("skills", [])])

    S = "pilot"
    check(S, "tutor opens printable check (12 items)", len(js(call(tut, "GET", "/api/pilot/form")).get("items", [])) == 12)
    call(tut, "PATCH", f"/api/kids/{kid}", {"study_code": "P01"})
    call(tut, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "pre", "score": 6, "out_of": 12})
    call(tut, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "post", "score": 9, "out_of": 12})
    csv = call(admin, "GET", "/api/pilot/export.csv")[1].decode()
    check(S, "admin downloads de-identified CSV with the gain", "P01" in csv and "25.0" in csv and "Mia" not in csv)

    S = "email"
    st = js(call(admin, "GET", "/api/admin/email")).get("mail", {})
    check(S, "email status shows outbox mode until set up", st.get("ready") is False or st.get("ready") is True, st)
    t = js(call(admin, "POST", "/api/admin/email/test"))
    check(S, "test email goes to admin (or test outbox)", t.get("status") in ("outbox", "sent"), t)
    prev = js(call(admin, "POST", "/api/admin/reports/preview"))
    check(S, "tonight's preview: one parent email for Mia", [p["kid"] for p in prev.get("parents", [])] == ["Mia"], prev.get("parents"))
    check(S, "tonight's preview: technical email for the admin", any(a["to"] == "admin@test.local" for a in prev.get("admin", [])))
    import email as _email
    from email import policy as _policy
    emls = sorted((TMP / "outbox").glob("*mum*.eml"))
    msg = _email.message_from_bytes(emls[-1].read_bytes(), policy=_policy.default) if emls else None
    att = [a for a in msg.iter_attachments()] if msg else []
    check(S, "parent email has the daily PDF attached", att and att[0].get_content().startswith(b"%PDF"), [a.get_filename() for a in att])
    check(S, "parent email body names the child and count", msg is not None and "Mia" in msg.get_body(preferencelist=("plain",)).get_content())
    tech = sorted((TMP / "outbox").glob("*admin*.eml"))
    tmsg = _email.message_from_bytes(tech[-1].read_bytes(), policy=_policy.default) if tech else None
    tbody = tmsg.get_body(preferencelist=("plain",)).get_content() if tmsg else ""
    check(S, "technical email has no child names", tmsg is not None and "Mia" not in tbody and "Mia" not in str(tmsg["Subject"]))
    call(par, "PATCH", "/api/auth/prefs", {"email_reports": False})
    prev2 = js(call(admin, "POST", "/api/admin/reports/preview"))
    check(S, "parent's 'off' switch stops their email", prev2.get("parents") == [], prev2.get("parents"))
    call(par, "PATCH", "/api/auth/prefs", {"email_reports": True})

    S = "account"
    check(S, "wrong current password refused", call(par, "POST", "/api/auth/password", {"current": "nope", "new": "NewParent42"})[0] == 400)
    check(S, "parent changes password", call(par, "POST", "/api/auth/password", {"current": "ParentPass9", "new": "NewParent42"})[0] == 200)
    check(S, "old password no longer works", call(client(), "POST", "/api/auth/login", {"email": "mum@test.local", "password": "ParentPass9"})[0] == 401)
    check(S, "sign out works", call(tut, "POST", "/api/auth/logout")[0] == 200 and call(tut, "GET", "/api/kids")[0] == 401)

    S = "delete"
    rep_dir = TMP / "portal" / "reports" / f"kid_{kid}"
    had = rep_dir.exists()
    check(S, "parent deletes the child and everything kept", call(par, "DELETE", f"/api/kids/{kid}")[0] == 200)
    check(S, "child's reports folder is gone", had and not rep_dir.exists())
    check(S, "child's sessions are gone", call(admin, "GET", f"/api/kids/{kid}/progress")[0] == 404)
    check("server", "no server errors during the walkthrough", server_tracebacks() == 0, f"{server_tracebacks()} tracebacks")
finally:
    stop_server()

passed = sum(r["passed"] for r in RESULTS)
OUT = ROOT / "outputs" / "tests"; OUT.mkdir(parents=True, exist_ok=True)
json.dump({"when": time.strftime("%Y-%m-%d %H:%M"), "passed": passed, "total": len(RESULTS), "results": RESULTS},
          open(OUT / "features.json", "w"), indent=1)
print(f"\nFEATURES: {passed}/{len(RESULTS)} checks passed")
assert passed == len(RESULTS), "FEATURE SUITE FAILED - see FAIL lines above"
